In [ ]:
!pip install PyPDF2 google-generativeai

import PyPDF2
import google.generativeai as genai
from google.colab import drive, files


print("✅ Setup complete")

In [ ]:
from google.colab import userdata
API_KEY = ""

# Configure Gemini
genai.configure(api_key=API_KEY)
model = genai.GenerativeModel('gemini-3-flash-preview')

# Test connection
response = model.generate_content("Say 'API is ready'")
print(response.text)

API is ready


In [ ]:
def extract_story_from_pdf(pdf_path, output_path="story.txt"):
    """
    Extract raw text from PDF and let Gemini clean it
    """
    # Step 1: Extract raw text from PDF
    print("📄 Reading PDF...")
    raw_text = ""
    with open(pdf_path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            raw_text += page.extract_text() + "\n\n"

    print(f"✓ Extracted {len(raw_text)} characters")

    # Step 2: Let Gemini clean and extract only the story
    print("🤖 Asking Gemini to extract the story...")

    prompt = f"""
    Extract ONLY the main story content from this book text.

    REMOVE these completely:
    - Table of contents
    - Foreword, preface, introduction
    - Copyright, publisher info, ISBN
    - Page numbers, headers, footers
    - Appendix, glossary, index
    - About the author, ads, reviews

    KEEP only:
    - The actual narrative/story from Chapter 1 to The End
    - All dialogue, descriptions, and narration
    - Original paragraph structure and indentation

    Return ONLY the cleaned story text, nothing else.

    TEXT TO CLEAN:
    {raw_text}
    """

    response = model.generate_content(prompt)
    story = response.text

    # Step 3: Save to file
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(story)

    print(f"✓ Saved {len(story)} characters to {output_path}")
    print(f"✓ Word count: {len(story.split())}")

    return story

print("✅ Function defined")

✅ Function defined


In [ ]:
# Upload PDF file
print("📤 Please upload your PDF file:")
uploaded = files.upload()

# Get the filename
pdf_filename = list(uploaded.keys())[0]
print(f"✅ Uploaded: {pdf_filename}")

📤 Please upload your PDF file:


Saving chapter1.pdf to chapter1.pdf
✅ Uploaded: chapter1.pdf


In [ ]:
# Run the extraction
story_text = extract_story_from_pdf(pdf_filename, "cleaned_story.txt")

# Preview
print("\n" + "="*50)
print("PREVIEW (first 500 characters):")
print("="*50)
print(story_text[:500])
print("...")

📄 Reading PDF...
✓ Extracted 11473 characters
🤖 Asking Gemini to extract the story...
✓ Saved 11380 characters to cleaned_story.txt
✓ Word count: 2191

PREVIEW (first 500 characters):
CHAPTER I. DOWN THE RABBIT-HOLE.  

ALICE was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it, “and what is the use of a book,” thought Alice, “without pictures or conversations?”  

So she was considering in her own mind, (as well as she could, for the hot day made her feel very sleepy and stupid,) whether the pleasure of making a daisy-
...


In [ ]:
# Download the cleaned story
files.download("cleaned_story.txt")
print("✅ File downloaded")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ File downloaded


In [ ]:
# ===== AGENT 1 TEST CELL: HARDCODED TEXT, SAME FUNCTION =====

# Hardcoded text to simulate PDF content
test_text = """
FOREWORD

This book is dedicated to all dreamers.

---

Chapter 1: The Beginning

Arjun stepped into the dark forest. The wind whispered through the trees.
"I have a bad feeling about this," he muttered.

A shadow moved.

---

APPENDIX

Extra notes about the world.

Afterword

Thanks for reading.
"""

# Save originals so we can restore them later
import builtins
original_open = builtins.open
original_pdf_reader = PyPDF2.PdfReader

# Fake page + fake reader
class FakePage:
    def extract_text(self):
        return test_text

class FakePdfReader:
    def __init__(self, file_obj):
        self.pages = [FakePage()]

# Fake open only for one dummy filename
class DummyFile:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        pass

def fake_open(path, mode='r', *args, **kwargs):
    if path == "dummy.pdf" and mode == "rb":
        return DummyFile()
    return original_open(path, mode, *args, **kwargs)

# Patch
builtins.open = fake_open
PyPDF2.PdfReader = FakePdfReader

try:
    print("🔹 Testing Agent 1 with hardcoded text through the same function...\n")

    # SAME FUNCTION CALL
    cleaned_story = extract_story_from_pdf("dummy.pdf", "test_story.txt")

    print("\n" + "="*60)
    print("✅ AGENT 1 OUTPUT")
    print("="*60)
    print(cleaned_story)

    print("\n📏 Length:", len(cleaned_story))
    print("📄 Words:", len(cleaned_story.split()))

finally:
    # Restore originals
    builtins.open = original_open
    PyPDF2.PdfReader = original_pdf_reader